# Madrid Metro Network

---

## Librerías

In [171]:
# Para correr en colab
try:
  from google.colab import output
  output.enable_custom_widget_manager()
  %pip -q install "plotly==5.24.1"
except ImportError:
  pass

In [172]:
from heapq import heappush, heappop
import plotly.graph_objects as go
import ipywidgets as widgets
import pandas as pd

---

## Dataset
La red, las coordenadas de los nodos y un diccionario para reconocer las estaciones de trasbordo entre rutas (pues en el dataframe original dichas intersecciones no comparten el mismo ID entre rutas).
Para ejecutar esto en Colab, se deben cargar primero los tres archivos:
- ```madrid_interchange_stops.csv```
- ```madrid_metro_network.csv```
- ```madrid_metro_stops.csv```

In [173]:
shared = pd.read_csv("madrid_interchange_stops.csv")
network = pd.read_csv("madrid_metro_network.csv")
nodes = pd.read_csv("madrid_metro_stops.csv")

---

## Tiempo mínimo
Se calcula el tiempo mínimo usando el algoritmo de Dijkstra. La penalización de 4 minutos por cambios de línea no es un problema trivial, pues el algoritmo de Dijkstra asume que los estados corresponden a mínimos globales. Sin embargo, esta penalización abre la posibilidad de que

- `A –›₁ B` < `A –›₂ B`, pero
- `A -›₁ B –›₂ C` > `A ->₂ B –›₂ C`

Por ello, en lugar de representar los estados como estaciones, los representamos como pares (estación, línea_de_llegada). Esto resuelve el problema al clasificar `A –›₁ B` y `A –›₂ B` como estados diferentes, pero incrementa la complejidad en tiempo y memoria a 

- $O(nk \; log₂(nk))$ temporal
- $O(nk)$ en memoria

Donde $n$ es el número de estaciones y $l$ el número de líneas distintas

In [174]:
class Dijkstra:
  def __init__(self, connections):
    '''Initializes with an adjacency list: [station] -> [(travelTime, arrivalStation, lineId), ...]'''
    self.connections = connections


  def __computeFrom(self, start):
    '''Runs Dijkstra from start. Heap node: (travelTime, currentStation, lastStation, currentLine, lastLine)'''
    distance = [{} for _ in self.connections]
    parent = [{} for _ in self.connections]
    heap = [(0, start, None, None, None)]

    while heap:
      shortestPath, currentStation, lastStation, currentLine, lastLine = heappop(heap)
      if currentLine not in distance[currentStation]:
        distance[currentStation][currentLine] = shortestPath
        parent[currentStation][currentLine] = (lastStation, lastLine)

        for travelTime, nextStation, nextLine in self.connections[currentStation]:
          totalTravelTime = shortestPath + travelTime
          if currentLine is not None and currentLine != nextLine:
            totalTravelTime += 4
          heappush(heap, (totalTravelTime, nextStation, currentStation, nextLine, currentLine))

    self.distance = distance
    self.parent = parent


  def __reconstructPath(self, station):
    '''Path: list of station indices. Directions: list of (station, lineId) pairs marking where to hop on and hop off'''
    self.directions = []
    self.path = []
    if not self.distance[station]:
      return

    line = min(self.distance[station], key=self.distance[station].get)
    self.directions.append((station, line))
    while station is not None:
      self.path.append(station)
      parentStation, parentLine = self.parent[station][line]

      if parentLine != line:
        self.directions.append((parentStation, line))
        if parentLine is not None:
          self.directions.append((parentStation, parentLine))
          
      station, line = parentStation, parentLine
    self.directions.reverse()
    self.path.reverse()


  def findShortestPath(self, start, end):
    '''Returns the minimum travel time and its path from start to end'''
    self.__computeFrom(start)
    self.__reconstructPath(end)
    return min(self.distance[end].values(), default = None), self.path

---

## Representación de la red
Se crea una lista de adyacencias con base en los archivos .csv y se inicializa un objeto para calcular el tiempo mínimo entre dos estaciones.

In [175]:
idToIndex = {node.id: index for index, node in enumerate(nodes.itertuples())}
for node in shared.itertuples(): 
  idToIndex[node.station_id] = idToIndex[node.intersection_id]

connections = [[] for _ in range(len(nodes))]

for travel in network.itertuples():
  departureStation = idToIndex[travel.from_stop_id]
  arrivalStation = idToIndex[travel.to_stop_id]
  travelTime = travel.mean_travel_time_min
  lineId = travel.route_id
  
  connections[departureStation].append((travelTime, arrivalStation, lineId))

---

## Detalles para VSC
Se aplica un fondo transparente para que Visual Studio Code no agregue un bloque blanco detrás de la figura.

In [176]:
from IPython.display import display, HTML

display(HTML('''
<style>
  .cell-output-ipywidget-background {
    background-color: transparent !important;
  }
  .jp-OutputArea-output {
    background-color: transparent !important;
  }
</style>
'''))

---

## Representación visual

Creación del mapa con las rutas del metro de Madrid usando Plotly.

In [177]:
COLORS = {
  'MAD_L0': '#FAFAFA',
  'MAD_L1': '#4EB4E4',
  'MAD_L2': '#DE211A',
  'MAD_L3': '#FFCB1B',
  'MAD_L4': '#A35B30',
  'MAD_L5': '#76B123',
  'MAD_L6': '#727A80',
  'MAD_LR': '#1F4E9C'
}

### Visual de conexiones

In [178]:
metro_lines = []

for line, stops in network.groupby('route_id'):
  dx, dy = [], []
  
  for travel in stops.itertuples():
    departureStation = idToIndex[travel.from_stop_id]
    arrivalStation = idToIndex[travel.to_stop_id]
    dx += [nodes.x[departureStation], nodes.x[arrivalStation], None]
    dy += [nodes.y[departureStation], nodes.y[arrivalStation], None]
  
  metro_lines.insert(0, go.Scatter(
    x = dx, y = dy, mode = "lines", hoverinfo = "skip",
    line = {'color': COLORS[line], 'width': 3}
  ))

### Visual de estaciones

In [179]:
stations = go.Scatter(
    x = list(nodes.x), y = list(nodes.y), mode = "markers", 
    hovertext = nodes.name, hoverinfo = "text", 
    marker = {
      'size': 9, 
      'color': [COLORS[i[:6]] for i in nodes.id], 
      'line': {'color': '#111', 'width': 2}
    }
)

### Visual de selección y resultado

In [180]:
selected = go.Scatter(
  mode = "markers", hoverinfo = "skip", 
  marker = {'color': '#111', 'size': 15, 'line': {'color': '#FFF', 'width': 2}}
)
route = go.Scatter(
  mode = "lines", hoverinfo = "skip", 
  line = {'color': '#FFF', 'width': 8}
)

### Definición de figura y renderizado por capas

In [181]:
fig = go.FigureWidget(data = [route, *metro_lines, selected, stations], layout = go.Layout(
  xaxis = dict(range = [0, 1024], visible = False, fixedrange = True),
  yaxis = dict(range = [0, 1024], visible = False, fixedrange = True),
  width = 800, height = 800, margin = dict(l = 0, r = 0, t = 0, b = 0),
  plot_bgcolor = "#111", showlegend = False, hovermode = "closest"
))
route, *metro_lines, selected, stations = fig.data

---

## Etiquetas e interacciones

### Las etiquetas con la información de la ruta y la lógica para pintar la ruta óptima

In [182]:
class Render:
  def __init__(self, connections):
    '''Initializes with an adjacency list: [station] -> [(travelTime, arrivalStation, lineId), ...]'''
    self.dijkstra = Dijkstra(connections)
    self.out = widgets.Output()
    self.origin = None
    self.task = None
    
    self.legendBox = widgets.VBox(layout = widgets.Layout(
      width = '800px', min_height = '180px', padding = '8px 12px', border = '1px solid #444'
    ))


  def view(self):
    '''Builds the map, the route legend, and the output area'''
    mapArea = widgets.GridBox(
      [widgets.Box([fig], layout = widgets.Layout(grid_area = 'map')), self.__info()],
      layout = widgets.Layout(grid_template_areas = '"map"', width = '800px')
    )
    return widgets.VBox([mapArea, self.legendBox, self.out])


  def onStationClick(self, trace, points, selector):
    '''Handles station clicks: the first one sets the origin, the second one draws the route'''
    with self.out:
      if not points.point_inds:
        return
      station = points.point_inds[0]

      if self.origin is None:
        self.origin = station
        with fig.batch_update():
          route.x, route.y = [], []
          selected.x, selected.y = [nodes.x[station]], [nodes.y[station]]
        self.legendBox.children = []
        return

      time, path = self.dijkstra.findShortestPath(self.origin, station)
      if path:
        with fig.batch_update():
          selected.x = [nodes.x[self.origin], nodes.x[station]]
          selected.y = [nodes.y[self.origin], nodes.y[station]]
          route.x = list(nodes.x[path])
          route.y = list(nodes.y[path])
        self.__legend(time)
      self.origin = None


  def __badge(self, line):
    '''Colored label with the line number'''
    return widgets.HTML(
      f"<div style='background: {COLORS[line]}; color: #000; font-weight: bold; width: 24px; text-align: center; margin-right: 8px;'>{line[-1]}</div>"
    )


  def __info(self):
    '''Info box with the line colors and names (bottom right)'''
    names = network.groupby('route_id').route_long_name.first()
    rows = []
    for line, name in names.items():
      text = name.replace(' - ', ' -<br>')
      rows.append(widgets.HBox([
        self.__badge(line),
        widgets.HTML(f"<div style='color: #EEE; font-size: 13px; line-height: 15px; height: 36px; display: flex; flex-direction: column; justify-content: center;'>{text}</div>")
      ], layout = widgets.Layout(align_items = 'center')))
    return widgets.VBox(rows, layout = widgets.Layout(
      grid_area = 'map', align_self = 'flex-end', margin = '10px 10px 10px auto',
      padding = '6px 10px', border = '1px solid #AAA', width = '200px'
    ))



  def __legend(self, time):
    '''Shows the optimal route's total time and step-by-step directions'''
    directions, path = self.dijkstra.directions, self.dijkstra.path
    start, end = directions[0][0], directions[-1][0]
    white = lambda text: widgets.HTML(f"<span style='color: #FFF;'>{text}</span>")
    bold = lambda text: widgets.HTML(f"<b style='color: #FFF;'>{text}</b>")
    rows = [widgets.HBox(
      [bold(f'{nodes.name[start]} → {nodes.name[end]}'),
       bold(f'{time:.0f} min')],
      layout = widgets.Layout(justify_content = 'space-between')
    )]

    cursor = 0
    for (rideStart, line), (rideEnd, _) in zip(directions[::2], directions[1::2]):
      stops = path.index(rideEnd, cursor) - cursor
      cursor += stops
      rows.append(widgets.HBox([
        self.__badge(line),
        white(f'{nodes.name[rideStart]} → {nodes.name[rideEnd]}  ·  {stops} stop{"s" * (stops != 1)}')
      ]))

    self.legendBox.children = rows

---

## Renderizado

In [183]:
render = Render(connections)
stations.on_click(render.onStationClick)
display(render.view())

    'data': [{'hoverinfo': 'skip',
             …